## **Imports**

In [1]:
import time

import cv2
from matplotlib import pyplot as plt
import numpy as np

!python -m pip install -q ultralytics
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 3.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## **Running on Video**

In [2]:
def process_clip(cap, model, fname):
    w,h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"(W,H): ({w},{h})")
    print(f"fps: {fps}")
    print(f"Total frames: {total}")

    start = time.time()
    frame_ctr = 0

    out = cv2.VideoWriter(fname,
                          cv2.VideoWriter_fourcc(*"mp4v"),
                          fps, (w, h))

    while cap.isOpened():
        ret, frame = cap.read() 
        if not ret: break

        # Filter results now by the vehicle classes we require
        results = model.predict(frame,
                                classes=[2,3,5,7],
                                conf=0.25,  # default conf threshold we settled with
                                agnostic_nms=True, # handles overlapping boxes with different classes
                                verbose=False)

        annot_res = results[0].plot(font_size=17, pil=True)
        out.write(cv2.cvtColor(np.array(annot_res), cv2.COLOR_RGB2BGR))
        frame_ctr += 1

        if frame_ctr % 200 == 0: 
            print(f"{frame_ctr}/{total} processed")

    elapsed = time.time() - start
    cap.release()
    out.release()
    
    print(f"Duration: {elapsed:.1f}s")
    print(f"FPS: {frame_ctr/elapsed:.1f}")


def playback(cap):
    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        cv2.imshow("VideoPlayback", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

In [3]:
model = YOLO("yolov8m.pt")

# First clip with overhead camera and heavy traffic density
cap2 = cv2.VideoCapture("../data/Dev_Clips/dev_overhead_heavy_01.mp4")
process_clip(cap2, model, "../outputs/output_detect_overhead_heavy_01.mp4")

# cap_out = cv2.VideoCapture("../data/Dev_Clips/dev_overhead_heavy_01.mp4")
# playback(cap_out)

(W,H): (3840,2160)
fps: 25.0
Total frames: 3000
200/3000 processed
400/3000 processed
600/3000 processed
800/3000 processed
1000/3000 processed
1200/3000 processed
1400/3000 processed
1600/3000 processed
1800/3000 processed
2000/3000 processed
2200/3000 processed
2400/3000 processed
2600/3000 processed
2800/3000 processed
3000/3000 processed
Duration: 976.0s
FPS: 3.1


**Observations on Annotated Clip**

**Note:** The annotated output clip produced in this run is available [here](https://youtu.be/ur29LWk4LOU). The frame numbers, timestamps, and other detection results referenced in the observations below are from this specific clip. Running the same code may produce slightly different results due to differences in hardware.

- Overhead camera with heavy traffic (exceeding the project-defined threshold of 17+ vehicles/lane/minute)
- **Clip processing speed:** 3.1 FPS on a single T4 GPU; 4K 3840×2160 input using YOLO v8m, 3000 frames total
- Confidence scores increase predictably as vehicles move closer to the camera
- The *reliable detection zone* is a narrow band close to the camera. Roughly the last 1-2 seconds before vehicles exit the camera's view at typical speed
- In this zone, vehicles appear large in the frame and are tightly-boxed with high-confidence scores.
- Vehicles pass through the reliable detection zone quickly (1-2 sec) because they’re close and moving fast relative to the frame
- Vehicles entering the reliable detection zone typically already carry a confidence score above 0.7
- Vehicles near the camera consistently reach 0.9 confidence just before they pass through
- Detection dropout (oscillation between box and no box) is common for far vechicles as expected
- Detection dropout, however, often disappears as the vehicle gets closer towards the camera (example at frames 500-750)
- Class flicker is present at close range for vehicle types that are ambigous (pickups, vans). But this should not affect total vehicle count  
- `agnostic_nms=True` is applied in inference this time so we did not encounter the same tightly overlapping different-class boxes we saw in the earlier static frame analyis
- Despite agnostic NMS, we can still observe split detections in articulated vehicles (car w. carrier, car hauler), especially as they drew near. Examples in frames 925-1025 and 1625-1725.
- However, unlike in the previous `detection-per-frame-analysis` notebook, these split detection boxes have low overlap (e.g., one box bounds the car while the other bounds the towed carrier)
- No hallucinations observed across the full clip at conf=0.25.

In [4]:
# Second clip roadside-mounted camera, heavy traffic, and congestion
cap2 = cv2.VideoCapture("../data/Dev_Clips/dev_roadside_heavy_congest_01.mp4")
process_clip(cap2, model, "../outputs/output_detect_roadside_heavy_congest_01.mp4")

# cap_out = cv2.VideoCapture("../data/Dev_Clips/dev_roadside_heavy_congest_01.mp4")
# playback(cap_out)

(W,H): (1920,1080)
fps: 25.0
Total frames: 2301
200/2301 processed
400/2301 processed
600/2301 processed
800/2301 processed
1000/2301 processed
1200/2301 processed
1400/2301 processed
1600/2301 processed
1800/2301 processed
2000/2301 processed
2200/2301 processed
Duration: 243.6s
FPS: 9.3


**Observations on Annotated Clip**

**Note:** The annotated output clip produced in this run is available [here](https://youtu.be/NTVVJdfpCCM). The frame numbers, timestamps, and other detection results referenced in the observations below are from this specific clip. Running the same code may produce slightly different results due to differences in hardware.

- Camera mounted roadside angled down. Showcases heavy traffic with segment of bumper-to-bumber congestion at the end.
- **Clip processing speed:** 9.3 FPS via single T4 GPU; 1920x1080px input on YOLOv8m, 2301 total frames.
- As with the overhead clip, confidence scores increase with a vehicle's proximity to the camera.
- The *reliable detection zone* here covers a larger portion of the visible road. Vehicles just coming into view are already large.
- Traffic in the incoming lanes have a slower pace. The congestion near the end of the clip shows overlapping vehicles with many detections degrading from occlusion
- Since detection quality degrades from occlusion, distance is no longer the primary determinant of confidence score and localization accuracy
- Without vehicle overlap and occlusion, near-camera vehicles still score high around 0.9, decreasing as distance increases
- Detection dropout is still common in far vehicles as seen throughout the clip
- Class flickers can also be observed, beginning at the first few seconds of the annotated clip (two trucks approaching oscillates between truck and bus)
- We can also observe split detections (example instances seen at frames 375-625) but often disappearing as the vehicle gets closer
- No hallucinations but non-vehicle misclassifications are seen. We can catch road markings being boxed as a car at some frames throughout the video.

## **Detection Instabilities Guide**

| Term | What happens | Example |
|------|--------------|---------|
| Detection dropout | Vehicle has a box, loses it for some frames, gets it back | Far truck detected at frame 100, gone 101–103, back at 104 |
| Class flicker | Box stays but label switches between frames | Same vehicle toggling car/truck frame to frame |
| Split detection | One object is fragmented into two or more boxes intermittently | Truck with trailer oscillates between one box and two |
| Merged detection | Two nearby objects are combined into one box | Two bumper-to-bumper cars boxed as a single detection at distance |
| Hallucination | Box drawn on literally nothing | Shadow or road marking boxed |
| False positive (task-relative) | Correct detection, wrong object class for our task | Person or traffic sign boxed. Real object but not a vehicle |
| Misclassification | Real vehicle but wrong vehicle label | Pickup truck labeled as car |

- We can find more accurate, stronger terminologies and explanations, but for this experiment, this should be enough to serve as a "quick look" guide.
- As observed in the clips, all these instabilities (except hallucination and false positive) are worse at distance, where the detector has to work with fewer pixels.
- Placing the counting line inside the reliable detection zone is key, allowing us to mitigate the negative impact of most of these when it comes to accurate counting.

## **Should We Fine Tune the Model?**

**Short answer:** Not right now.

**Long answer:** The pretrained YOLOv8m on COCO weights is doing well enough for our pipeline. Inside the reliable detection zone (where counting must happen), vehicles are consistently detected having confidence scores above 0.7 with tight boxes. Detection dropout is practically nonexistent in the detection zone except with real overlapping objects and occlusions (exacerbated by `agnostic_nms=True`). We also saw zero hallucinations across the two dev clips at the default 0.25 confidence threshold. Yes, there are imperfections. Pickups flicker between car/truck and vans may get labeled as also either depending on the angle. But for a total vehicle count, misclassification between vehicle classes are not too impactful, since a truck labeled as a car is still a vehicle that will be counted. The real flicker issues (dropout and class switching) are often found in the far zone where vehicles are small and hard to resolve. But we’re not counting there. Our counting line will sit inside the reliable zone. In additon, fine-tuning is time expensive. We will have to collect frames, draw bounding boxes, annotate ground truths, and train and validate. At this point, I am unsure if the payoff would signficant or simply marginal given how well pretrained can perform. If our predicted counts end up significantly off from hand counts later, and we can trace that gap back to detection failures (not tracker issues), we’ll revisit. Until then, pretrained for now.